# Ablation Study with All Baseline ML Models for UAV Propeller Fault Diagnosis

This notebook performs an ablation study using all classical baseline machine learning models.

## Baseline models included

1. Random Forest  
2. SVM with RBF kernel  
3. XGBoost  
4. KNN  
5. Logistic Regression  

## Ablation settings

The notebook compares:

1. Measured features only  
2. Physics-informed features only  
3. Measured + physics-informed features  
4. With non-overlapping time-windowing  
5. Without time-windowing  

## Validation

The notebook uses `source_file` as the group variable for leakage-controlled validation.

## Input file

```text
merged_physics.csv
```

The file must contain:

- `class_label`
- `source_file`
- measured features
- physics-informed features


In [1]:
# ============================================================
# Ablation Study using All Baseline ML Models
# UAV Propeller Fault Diagnosis
# Models: RF, SVM, XGBoost, KNN, Logistic Regression
# ============================================================

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as e:
    XGBOOST_AVAILABLE = False
    print("XGBoost is not installed. Install it using: pip install xgboost")
    print("Error:", e)

# --------------------------
# Global settings
# --------------------------
SEED = 42
np.random.seed(SEED)

DATA_FILE = "merged_physics.csv"
OUTPUT_DIR = Path("Results_Ablation_All_Baseline_ML")

WINDOW_SIZE = 50
STRIDE = 50
FILTER_ACTIVE_REGION = True
ACTIVE_RPM_MIN = 500
ACTIVE_ESC_MIN = 1100

N_SPLITS = 3

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------
# Plot settings
# --------------------------
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 14
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["legend.fontsize"] = 12
plt.rcParams["xtick.labelsize"] = 12
plt.rcParams["ytick.labelsize"] = 12
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

print("Configuration loaded.")
print("Input file:", DATA_FILE)
print("Output folder:", OUTPUT_DIR)


Configuration loaded.
Input file: merged_physics.csv
Output folder: Results_Ablation_All_Baseline_ML


In [2]:
# ============================================================
# Helper functions
# ============================================================

def find_col(df, possible_names, required=True):
    """Find a column by exact or partial matching."""
    col_map = {c.lower().strip(): c for c in df.columns}

    for name in possible_names:
        key = name.lower().strip()
        if key in col_map:
            return col_map[key]

    for c in df.columns:
        c_low = c.lower().strip()
        for name in possible_names:
            if name.lower().strip() in c_low:
                return c

    if required:
        raise ValueError(f"Could not find any of these columns: {possible_names}")
    return None


def majority_label(labels):
    """Return majority label from an array."""
    values, counts = np.unique(labels, return_counts=True)
    return values[np.argmax(counts)]


def compute_metrics(y_true, y_pred, all_labels, class_names):
    """Compute macro, weighted, and per-class recall metrics."""
    out = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_macro": precision_score(y_true, y_pred, labels=all_labels, average="macro", zero_division=0),
        "Recall_macro": recall_score(y_true, y_pred, labels=all_labels, average="macro", zero_division=0),
        "F1_macro": f1_score(y_true, y_pred, labels=all_labels, average="macro", zero_division=0),
        "Precision_weighted": precision_score(y_true, y_pred, labels=all_labels, average="weighted", zero_division=0),
        "Recall_weighted": recall_score(y_true, y_pred, labels=all_labels, average="weighted", zero_division=0),
        "F1_weighted": f1_score(y_true, y_pred, labels=all_labels, average="weighted", zero_division=0),
    }

    per_class_recall = recall_score(
        y_true,
        y_pred,
        labels=all_labels,
        average=None,
        zero_division=0
    )

    for cls_name, rec in zip(class_names, per_class_recall):
        safe_name = str(cls_name).replace(" ", "_").replace("-", "_")
        out[f"Recall_{safe_name}"] = rec

    return out


def save_confusion_matrix(y_true, y_pred, labels, class_names, out_pdf, title):
    """Save confusion matrix as PDF."""
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6.8, 5.8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Predicted Label", fontweight="bold")
    ax.set_ylabel("True Label", fontweight="bold")
    plt.tight_layout()
    fig.savefig(out_pdf, format="pdf", bbox_inches="tight")
    plt.close(fig)


def save_bar_plot(df, out_pdf, title, metric_col="F1_macro_Mean"):
    """Save comparison bar plot as PDF."""
    plot_df = df.copy().sort_values(metric_col, ascending=True)
    labels = plot_df["Experiment"].astype(str) + " | " + plot_df["Model"].astype(str)

    fig_height = max(6, 0.32 * len(plot_df))
    fig, ax = plt.subplots(figsize=(12, fig_height))
    ax.barh(labels, plot_df[metric_col])
    ax.set_xlabel(metric_col.replace("_", " "), fontweight="bold")
    ax.set_ylabel("Experiment and Model", fontweight="bold")
    ax.set_title(title, fontweight="bold")
    ax.grid(axis="x", linestyle="--", alpha=0.35)
    plt.tight_layout()
    fig.savefig(out_pdf, format="pdf", bbox_inches="tight")
    plt.close(fig)


print("Helper functions ready.")


Helper functions ready.


In [3]:
# ============================================================
# Load dataset and prepare feature groups
# ============================================================

df = pd.read_csv(DATA_FILE)
df.columns = df.columns.str.strip()

label_col = find_col(df, ["class_label", "label", "class"])
group_col = find_col(df, ["source_file", "source", "file", "run", "group"])

rpm_col = find_col(df, ["Motor Electrical Speed (RPM)", "RPM", "Motor Speed"], required=False)
esc_col = find_col(df, ["ESC signal (µs)", "ESC signal", "ESC"], required=False)

df[label_col] = df[label_col].astype(str).str.strip()
df[group_col] = df[group_col].astype(str).str.strip()

# Convert numerical columns
for col in df.columns:
    if col not in [label_col, group_col]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove idle region
if FILTER_ACTIVE_REGION:
    if rpm_col is not None:
        df = df[df[rpm_col] > ACTIVE_RPM_MIN].copy()
        print(f"Active-region filtering applied: {rpm_col} > {ACTIVE_RPM_MIN}")
    elif esc_col is not None:
        df = df[df[esc_col] >= ACTIVE_ESC_MIN].copy()
        print(f"Active-region filtering applied: {esc_col} >= {ACTIVE_ESC_MIN}")
    else:
        print("Warning: No RPM or ESC column found. Active-region filtering skipped.")

# Numerical feature columns
all_numeric_cols = [
    c for c in df.columns
    if c not in [label_col, group_col] and pd.api.types.is_numeric_dtype(df[c])
]

# Identify physics-informed columns by expected names or patterns
physics_keywords = [
    "calculated",
    "relative error",
    "power error",
    "angular speed",
    "thrust-to-rpm",
    "torque-to-rpm",
    "thrust per electrical watt",
    "thrust_per_watt",
    "electrical_power_calculated",
    "mechanical_power_calculated",
    "electrical_power_error",
    "mechanical_power_error",
    "rpm_squared",
    "rpm2",
    "physics"
]

physics_cols = []
for c in all_numeric_cols:
    c_low = c.lower()
    if any(k in c_low for k in physics_keywords):
        physics_cols.append(c)

# Fallback expected physics feature names
expected_physics_cols = [
    "Electrical Power Calculated (W)",
    "Electrical Power Error (W)",
    "Electrical Power Relative Error",
    "Angular Speed (rad/s)",
    "Mechanical Power Calculated (W)",
    "Mechanical Power Error (W)",
    "Mechanical Power Relative Error",
    "Thrust-to-RPM Ratio (gf/RPM)",
    "Torque-to-RPM Ratio (N·m/RPM)",
    "Thrust per Electrical Watt (gf/W)",
    "Thrust-to-RPM-Squared Ratio (gf/RPM^2)",
    "Torque-to-RPM-Squared Ratio (N·m/RPM^2)"
]

for c in expected_physics_cols:
    if c in all_numeric_cols and c not in physics_cols:
        physics_cols.append(c)

measured_cols = [c for c in all_numeric_cols if c not in physics_cols]

if len(physics_cols) == 0:
    raise ValueError(
        "No physics-informed columns were detected. "
        "Please check column names or manually define physics_cols."
    )

if len(measured_cols) == 0:
    raise ValueError(
        "No measured columns were detected. "
        "Please check column names or manually define measured_cols."
    )

df = df.dropna(subset=all_numeric_cols + [label_col, group_col]).copy()

print("Dataset shape after preprocessing:", df.shape)
print("Label column:", label_col)
print("Group column:", group_col)
print("Measured feature count:", len(measured_cols))
print("Physics-informed feature count:", len(physics_cols))
print("All feature count:", len(all_numeric_cols))

print("\nDetected physics-informed columns:")
for c in physics_cols:
    print(" -", c)

print("\nClass distribution:")
print(df[label_col].value_counts())

print("\nNumber of source-file groups:", df[group_col].nunique())


Active-region filtering applied: Motor Electrical Speed (RPM) > 500
Dataset shape after preprocessing: (1396, 33)
Label column: class_label
Group column: source_file
Measured feature count: 23
Physics-informed feature count: 8
All feature count: 31

Detected physics-informed columns:
 - Electrical Power Calculated (W)
 - Electrical Power Error (W)
 - Electrical Power Relative Error
 - Angular Speed (rad/s)
 - Mechanical Power Calculated (W)
 - Mechanical Power Error (W)
 - Mechanical Power Relative Error
 - Thrust per Electrical Watt (gf/W)

Class distribution:
class_label
bend      591
normal    589
crack     216
Name: count, dtype: int64

Number of source-file groups: 9


In [4]:
# ============================================================
# Define all baseline ML models
# ============================================================

models = {
    "Random_Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        class_weight="balanced",
        n_jobs=-1
    ),

    "SVM_RBF": SVC(
        kernel="rbf",
        C=10.0,
        gamma="scale",
        class_weight="balanced",
        probability=True,
        random_state=SEED
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5,
        weights="distance",
        metric="minkowski"
    ),

    "Logistic_Regression": LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    )
}

if XGBOOST_AVAILABLE:
    models["XGBoost"] = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1
    )

print("Baseline ML models included:")
for name in models:
    print(" -", name)


Baseline ML models included:
 - Random_Forest
 - SVM_RBF
 - KNN
 - Logistic_Regression
 - XGBoost


In [5]:
# ============================================================
# Data construction functions
# ============================================================

def make_non_windowed_data(df, feature_cols, label_col, group_col):
    """Use each row as one sample."""
    X = df[feature_cols].values.astype(np.float32)
    y_str = df[label_col].values
    groups = df[group_col].values
    return X, y_str, groups


def make_windowed_data(df, feature_cols, label_col, group_col, window_size=50, stride=50):
    """Create non-overlapping windows inside each source file and flatten for classical ML."""
    X_windows = []
    y_windows = []
    group_windows = []

    for group_name, gdf in df.groupby(group_col, sort=False):
        gdf = gdf.reset_index(drop=True)
        X_arr = gdf[feature_cols].values.astype(np.float32)
        y_arr = gdf[label_col].values

        n = len(gdf)
        if n < window_size:
            continue

        for start in range(0, n - window_size + 1, stride):
            end = start + window_size
            X_win = X_arr[start:end]
            y_win = majority_label(y_arr[start:end])

            X_windows.append(X_win.reshape(-1))
            y_windows.append(y_win)
            group_windows.append(group_name)

    X_windows = np.asarray(X_windows, dtype=np.float32)
    y_windows = np.asarray(y_windows)
    group_windows = np.asarray(group_windows)

    return X_windows, y_windows, group_windows


feature_sets = {
    "Measured_Only": measured_cols,
    "Physics_Only": physics_cols,
    "Measured_Plus_Physics": measured_cols + physics_cols
}

print("Feature sets:")
for k, v in feature_sets.items():
    print(k, ":", len(v), "features")


Feature sets:
Measured_Only : 23 features
Physics_Only : 8 features
Measured_Plus_Physics : 31 features


In [6]:
# ============================================================
# Group-wise cross-validation runner
# ============================================================

def run_group_kfold_experiment(X, y_str, groups, experiment_name, output_dir):
    """Run Stratified Group K-Fold evaluation for all baseline models."""
    exp_dir = output_dir / experiment_name
    exp_dir.mkdir(parents=True, exist_ok=True)

    le = LabelEncoder()
    y = le.fit_transform(y_str)

    class_names = le.classes_
    all_labels = np.arange(len(class_names))

    n_unique_groups = len(np.unique(groups))
    n_splits = min(N_SPLITS, n_unique_groups)

    if n_splits < 2:
        raise ValueError(f"Experiment {experiment_name}: At least 2 source-file groups are required.")

    sgkf = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=SEED
    )

    fold_rows = []
    pred_frames = []

    for fold_id, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups), start=1):
        fold_dir = exp_dir / f"Fold_{fold_id}"
        fold_dir.mkdir(parents=True, exist_ok=True)

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        split_info = {
            "experiment": experiment_name,
            "fold": fold_id,
            "train_groups": np.unique(groups[train_idx]).tolist(),
            "test_groups": np.unique(groups[test_idx]).tolist(),
            "n_train_samples": int(len(train_idx)),
            "n_test_samples": int(len(test_idx))
        }

        with open(fold_dir / "source_file_split.json", "w", encoding="utf-8") as f:
            json.dump(split_info, f, indent=4)

        for model_name, clf in models.items():
            print(f"{experiment_name} | Fold {fold_id} | {model_name}")

            model_dir = fold_dir / model_name
            model_dir.mkdir(parents=True, exist_ok=True)

            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("clf", clf)
            ])

            pipe.fit(X_train, y_train)
            y_pred = pipe.predict(X_test)

            metrics = compute_metrics(y_test, y_pred, all_labels, class_names)
            metrics["Experiment"] = experiment_name
            metrics["Fold"] = fold_id
            metrics["Model"] = model_name
            metrics["N_Test"] = len(test_idx)
            fold_rows.append(metrics)

            report = classification_report(
                y_test,
                y_pred,
                labels=all_labels,
                target_names=class_names,
                digits=4,
                zero_division=0
            )

            with open(model_dir / "classification_report.txt", "w", encoding="utf-8") as f:
                f.write(report)

            save_confusion_matrix(
                y_true=y_test,
                y_pred=y_pred,
                labels=all_labels,
                class_names=class_names,
                out_pdf=model_dir / "confusion_matrix.pdf",
                title=f"{experiment_name} - {model_name} - Fold {fold_id}"
            )

            pred_df = pd.DataFrame({
                "Experiment": experiment_name,
                "Fold": fold_id,
                "Model": model_name,
                "group": groups[test_idx],
                "true_label": le.inverse_transform(y_test),
                "predicted_label": le.inverse_transform(y_pred)
            })
            pred_df.to_csv(model_dir / "predictions.csv", index=False)
            pred_frames.append(pred_df)

    fold_df = pd.DataFrame(fold_rows)
    pred_all_df = pd.concat(pred_frames, axis=0, ignore_index=True) if len(pred_frames) else pd.DataFrame()

    fold_df.to_csv(exp_dir / "fold_level_metrics.csv", index=False)
    pred_all_df.to_csv(exp_dir / "all_predictions.csv", index=False)

    metric_cols = [c for c in fold_df.columns if c not in ["Experiment", "Fold", "Model", "N_Test"]]
    summary_rows = []

    for model_name, gdf in fold_df.groupby("Model"):
        row = {
            "Experiment": experiment_name,
            "Model": model_name,
            "N_Folds": gdf["Fold"].nunique()
        }
        for metric in metric_cols:
            row[f"{metric}_Mean"] = gdf[metric].mean()
            row[f"{metric}_Std"] = gdf[metric].std(ddof=1) if len(gdf) > 1 else 0.0
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    summary_df = summary_df.sort_values("F1_macro_Mean", ascending=False)
    summary_df.to_csv(exp_dir / "mean_std_summary.csv", index=False)

    return fold_df, summary_df


print("Group-wise experiment runner ready.")


Group-wise experiment runner ready.


In [7]:
# ============================================================
# Run complete ablation study
# ============================================================

all_fold_results = []
all_summary_results = []

for feature_set_name, feature_cols in feature_sets.items():

    # --------------------------
    # Ablation without time-windowing
    # --------------------------
    X_nowin, y_nowin, groups_nowin = make_non_windowed_data(
        df,
        feature_cols,
        label_col,
        group_col
    )

    experiment_name = f"{feature_set_name}_Without_Windowing"

    fold_df, summary_df = run_group_kfold_experiment(
        X=X_nowin,
        y_str=y_nowin,
        groups=groups_nowin,
        experiment_name=experiment_name,
        output_dir=OUTPUT_DIR
    )

    all_fold_results.append(fold_df)
    all_summary_results.append(summary_df)

    # --------------------------
    # Ablation with non-overlapping time-windowing
    # --------------------------
    X_win, y_win, groups_win = make_windowed_data(
        df,
        feature_cols,
        label_col,
        group_col,
        window_size=WINDOW_SIZE,
        stride=STRIDE
    )

    experiment_name = f"{feature_set_name}_With_NonOverlapping_Windowing"

    fold_df, summary_df = run_group_kfold_experiment(
        X=X_win,
        y_str=y_win,
        groups=groups_win,
        experiment_name=experiment_name,
        output_dir=OUTPUT_DIR
    )

    all_fold_results.append(fold_df)
    all_summary_results.append(summary_df)

all_fold_results_df = pd.concat(all_fold_results, axis=0, ignore_index=True)
all_summary_results_df = pd.concat(all_summary_results, axis=0, ignore_index=True)

all_fold_results_df.to_csv(OUTPUT_DIR / "all_ablation_fold_level_metrics.csv", index=False)
all_summary_results_df.to_csv(OUTPUT_DIR / "all_ablation_mean_std_summary.csv", index=False)

print("Ablation study completed.")
print("Saved:", OUTPUT_DIR / "all_ablation_fold_level_metrics.csv")
print("Saved:", OUTPUT_DIR / "all_ablation_mean_std_summary.csv")

all_summary_results_df.head(20)


Measured_Only_Without_Windowing | Fold 1 | Random_Forest
Measured_Only_Without_Windowing | Fold 1 | SVM_RBF
Measured_Only_Without_Windowing | Fold 1 | KNN
Measured_Only_Without_Windowing | Fold 1 | Logistic_Regression
Measured_Only_Without_Windowing | Fold 1 | XGBoost
Measured_Only_Without_Windowing | Fold 2 | Random_Forest
Measured_Only_Without_Windowing | Fold 2 | SVM_RBF
Measured_Only_Without_Windowing | Fold 2 | KNN
Measured_Only_Without_Windowing | Fold 2 | Logistic_Regression
Measured_Only_Without_Windowing | Fold 2 | XGBoost
Measured_Only_Without_Windowing | Fold 3 | Random_Forest
Measured_Only_Without_Windowing | Fold 3 | SVM_RBF
Measured_Only_Without_Windowing | Fold 3 | KNN
Measured_Only_Without_Windowing | Fold 3 | Logistic_Regression
Measured_Only_Without_Windowing | Fold 3 | XGBoost
Measured_Only_With_NonOverlapping_Windowing | Fold 1 | Random_Forest
Measured_Only_With_NonOverlapping_Windowing | Fold 1 | SVM_RBF
Measured_Only_With_NonOverlapping_Windowing | Fold 1 | KNN
Me

,Experiment,Model,N_Folds,Accuracy_Mean,Accuracy_Std,Precision_macro_Mean,Precision_macro_Std,Recall_macro_Mean,Recall_macro_Std,F1_macro_Mean,...,Recall_weighted_Mean,Recall_weighted_Std,F1_weighted_Mean,F1_weighted_Std,Recall_bend_Mean,Recall_bend_Std,Recall_crack_Mean,Recall_crack_Std,Recall_normal_Mean,Recall_normal_Std
0,Measured_Only_Without_Windowing,Random_Forest,3,0.815331,0.222066,0.769948,0.198023,0.670908,0.290202,0.711342,...,0.815331,0.222066,0.881028,0.151288,0.509306,0.500260,0.875866,0.104454,0.627551,0.546633
1,Measured_Only_Without_Windowing,XGBoost,3,0.820950,0.195746,0.734175,0.237690,0.681497,0.279316,0.701026,...,0.820950,0.195746,0.878634,0.145163,0.524535,0.501803,0.906860,0.093836,0.613095,0.537002
2,Measured_Only_Without_Windowing,Logistic_Regression,3,0.941803,0.084660,0.748513,0.222174,0.687381,0.299444,0.698921,...,0.941803,0.084660,0.939760,0.096020,0.653130,0.565992,0.750000,0.433013,0.659014,0.570838
3,Measured_Only_Without_Windowing,SVM_RBF,3,0.898510,0.152101,0.755408,0.212893,0.668747,0.320159,0.689308,...,0.898510,0.152101,0.915907,0.132574,0.604907,0.531993,0.744872,0.428641,0.656463,0.568719
4,Measured_Only_Without_Windowing,KNN,3,0.834451,0.220121,0.744198,0.206277,0.628755,0.319373,0.668194,...,0.834451,0.220121,0.876354,0.169106,0.541455,0.503264,0.695974,0.318113,0.648835,0.562026
5,Measured_Only_With_NonOverlapping_Windowing,Logistic_Regression,3,0.904762,0.164957,0.777778,0.192450,0.740741,0.231296,0.755556,...,0.904762,0.164957,0.942857,0.098974,0.555556,0.509175,1.000000,0.000000,0.666667,0.577350
6,Measured_Only_With_NonOverlapping_Windowing,SVM_RBF,3,0.857143,0.247436,0.777778,0.192450,0.722222,0.254588,0.740741,...,0.857143,0.247436,0.904762,0.164957,0.500000,0.500000,1.000000,0.000000,0.666667,0.577350
7,Measured_Only_With_NonOverlapping_Windowing,Random_Forest,3,0.857143,0.142857,0.638889,0.048113,0.629630,0.064150,0.628571,...,0.857143,0.142857,0.874830,0.109623,0.555556,0.509175,0.666667,0.577350,0.666667,0.577350
8,Measured_Only_With_NonOverlapping_Windowing,XGBoost,3,0.809524,0.082479,0.638889,0.048113,0.611111,0.055556,0.618470,...,0.809524,0.082479,0.848856,0.065480,0.611111,0.535758,0.666667,0.577350,0.555556,0.509175
9,Measured_Only_With_NonOverlapping_Windowing,KNN,3,0.380952,0.540848,0.388889,0.535758,0.444444,0.509175,0.407407,...,0.380952,0.540848,0.365079,0.551916,0.333333,0.577350,0.666667,0.577350,0.333333,0.577350


In [8]:
# ============================================================
# Save compact comparison plots
# ============================================================

plot_df = all_summary_results_df.copy()

save_bar_plot(
    plot_df,
    OUTPUT_DIR / "ablation_f1_macro_comparison.pdf",
    "Ablation Study using All Baseline ML Models",
    metric_col="F1_macro_Mean"
)

save_bar_plot(
    plot_df,
    OUTPUT_DIR / "ablation_accuracy_comparison.pdf",
    "Ablation Study using All Baseline ML Models",
    metric_col="Accuracy_Mean"
)

print("Plots saved:")
print(OUTPUT_DIR / "ablation_f1_macro_comparison.pdf")
print(OUTPUT_DIR / "ablation_accuracy_comparison.pdf")


Plots saved:
Results_Ablation_All_Baseline_ML\ablation_f1_macro_comparison.pdf
Results_Ablation_All_Baseline_ML\ablation_accuracy_comparison.pdf


In [9]:
# ============================================================
# Generate LaTeX table for manuscript
# ============================================================

def make_latex_ablation_table(summary_df, out_file):
    """Create a compact LaTeX table using Accuracy, Macro Recall, Macro F1, and Weighted F1."""
    df_ltx = summary_df.copy()
    df_ltx = df_ltx.sort_values(["Experiment", "F1_macro_Mean"], ascending=[True, False])

    lines = []
    lines.append(r"\begin{table*}[!htbp]")
    lines.append(r"\centering")
    lines.append(r"\scriptsize")
    lines.append(r"\caption{Ablation study using all baseline machine learning models under Stratified Group K-Fold validation.}")
    lines.append(r"\label{tab:baseline_ml_ablation}")
    lines.append(r"\resizebox{\textwidth}{!}{")
    lines.append(r"\begin{tabular}{llcccc}")
    lines.append(r"\hline")
    lines.append(r"\textbf{Ablation Setting} & \textbf{Model} & \textbf{Accuracy} & \textbf{Macro Recall} & \textbf{Macro F1} & \textbf{Weighted F1} \\")
    lines.append(r"\hline")

    for _, row in df_ltx.iterrows():
        exp = str(row["Experiment"]).replace("_", r"\_")
        model = str(row["Model"]).replace("_", r"\_")
        acc = f"{row['Accuracy_Mean']:.3f} $\\pm$ {row['Accuracy_Std']:.3f}"
        rec = f"{row['Recall_macro_Mean']:.3f} $\\pm$ {row['Recall_macro_Std']:.3f}"
        f1 = f"{row['F1_macro_Mean']:.3f} $\\pm$ {row['F1_macro_Std']:.3f}"
        wf1 = f"{row['F1_weighted_Mean']:.3f} $\\pm$ {row['F1_weighted_Std']:.3f}"
        lines.append(f"{exp} & {model} & {acc} & {rec} & {f1} & {wf1} \\\\")

    lines.append(r"\hline")
    lines.append(r"\end{tabular}")
    lines.append(r"}")
    lines.append(r"\end{table*}")

    latex_text = "\n".join(lines)

    with open(out_file, "w", encoding="utf-8") as f:
        f.write(latex_text)

    return latex_text


latex_table = make_latex_ablation_table(
    all_summary_results_df,
    OUTPUT_DIR / "baseline_ml_ablation_latex_table.txt"
)

print(latex_table)


\begin{table*}[!htbp]
\centering
\scriptsize
\caption{Ablation study using all baseline machine learning models under Stratified Group K-Fold validation.}
\label{tab:baseline_ml_ablation}
\resizebox{\textwidth}{!}{
\begin{tabular}{llcccc}
\hline
\textbf{Ablation Setting} & \textbf{Model} & \textbf{Accuracy} & \textbf{Macro Recall} & \textbf{Macro F1} & \textbf{Weighted F1} \\
\hline
Measured\_Only\_With\_NonOverlapping\_Windowing & Logistic\_Regression & 0.905 $\pm$ 0.165 & 0.741 $\pm$ 0.231 & 0.756 $\pm$ 0.214 & 0.943 $\pm$ 0.099 \\
Measured\_Only\_With\_NonOverlapping\_Windowing & SVM\_RBF & 0.857 $\pm$ 0.247 & 0.722 $\pm$ 0.255 & 0.741 $\pm$ 0.231 & 0.905 $\pm$ 0.165 \\
Measured\_Only\_With\_NonOverlapping\_Windowing & Random\_Forest & 0.857 $\pm$ 0.143 & 0.630 $\pm$ 0.064 & 0.629 $\pm$ 0.034 & 0.875 $\pm$ 0.110 \\
Measured\_Only\_With\_NonOverlapping\_Windowing & XGBoost & 0.810 $\pm$ 0.082 & 0.611 $\pm$ 0.056 & 0.618 $\pm$ 0.018 & 0.849 $\pm$ 0.065 \\
Measured\_Only\_With\_NonOver